## SHAP explanations (same test examples as LIME)

This notebook computes SHAP (KernelExplainer) values for the **same** test indices used in LIME.

Reproducibility notes:
- We fix `SEED` and persist both the chosen test indices and the chosen background indices.
- KernelExplainer is sampling-based; we set NumPy's seed before each explanation.

Outputs are saved to `artifacts/shap/`:
- `chosen_test_indices.json` (copied from LIME)
- `background_indices.json`
- `shap_values_class0.npy`, `shap_values_class1.npy`
- `expected_value.json`
- `summary.json`
- per-instance plots (`waterfall_*.png`)


In [1]:
import sys

# Install SHAP into the *current* Jupyter kernel environment.
# If installation succeeds but import still fails, restart the kernel and re-run.
!"{sys.executable}" -m pip install -q --upgrade pip
!"{sys.executable}" -m pip install -q shap matplotlib
print("Python:", sys.executable)


Python: c:\Users\bestiary\AppData\Local\Programs\Python\Python312\python.exe


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.16.2 requires keras>=3.0.0, which is not installed.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.2.6 which is incompatible.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 6.33.1 which is incompatible.
tensorflow-intel 2.16.2 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-intel 2.16.2 requires numpy<2.0.0,>=1.26.0; python_version >= "3.12", but you have numpy 2.2.6 which is incompatible.
tensorflow-intel 2.16.2 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.1 which is incompatible.
tensorflow-intel 2.16.2 requires tensorboard<2.17,>=2.16, but you have tensorboard 2.20.0 which i

In [2]:
from pathlib import Path
import json

import joblib
import numpy as np
import matplotlib.pyplot as plt
import shap



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\bestiary\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\bestiary\AppData\Roaming\Python\Python312\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\bestiary\AppData\Roaming\Python\Python312\site-packages\ipykernel\kernelapp.py", line 758, in start
    s

AttributeError: _ARRAY_API not found

In [3]:
SEED = 42
NSAMPLES = 5000  # SHAP sampling budget per instance (increase for more stability)
K_TOP = 10       # for compact display/saving
BACKGROUND_SIZE = 50

DATA_DIR = Path("data_processed")
MLP_DIR = Path("artifacts/mlp")
LIME_DIR = Path("artifacts/lime")
OUT_DIR = Path("artifacts/shap")
OUT_DIR.mkdir(parents=True, exist_ok=True)

X_train = np.load(DATA_DIR / "X_train.npy")
y_train = np.load(DATA_DIR / "y_train.npy")
X_test = np.load(DATA_DIR / "X_test.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

meta = json.loads((DATA_DIR / "metadata.json").read_text(encoding="utf-8"))
feature_names = meta["feature_names"]
class_names = meta["target_names"]

model = joblib.load(MLP_DIR / "model.joblib")

chosen = json.loads((LIME_DIR / "chosen_test_indices.json").read_text(encoding="utf-8"))
test_indices = [int(i) for i in chosen["chosen_test_indices"]]
len(test_indices), test_indices[:5]


(10, [7, 15, 49, 53, 55])

In [4]:
# Persist the test indices we are using (copied from LIME)
(OUT_DIR / "chosen_test_indices.json").write_text(
    json.dumps(chosen, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "chosen_test_indices.json").as_posix()


'artifacts/shap/chosen_test_indices.json'

In [5]:
# Choose a deterministic background set from X_train
rng = np.random.default_rng(SEED)
bg_idx = rng.choice(np.arange(X_train.shape[0]), size=min(BACKGROUND_SIZE, X_train.shape[0]), replace=False)
bg_idx = [int(i) for i in bg_idx]
background = X_train[bg_idx]

(OUT_DIR / "background_indices.json").write_text(
    json.dumps({"seed": SEED, "background_size": int(len(bg_idx)), "indices": bg_idx}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
background.shape


(50, 30)

In [6]:
def predict_proba_fn(X):
    return model.predict_proba(X)

# KernelExplainer for model-agnostic SHAP
explainer = shap.KernelExplainer(predict_proba_fn, background)
explainer.expected_value


array([0.27927476, 0.72072524])

In [7]:
# Save expected values
(OUT_DIR / "expected_value.json").write_text(
    json.dumps({"expected_value": [float(v) for v in explainer.expected_value]}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUT_DIR / "expected_value.json").as_posix()


'artifacts/shap/expected_value.json'

In [8]:
# Compute SHAP values for selected test instances
shap_class0 = []
shap_class1 = []
per_instance = []

for idx in test_indices:
    x = X_test[idx]
    true_y = int(y_test[idx])
    proba = predict_proba_fn(x.reshape(1, -1))[0]
    pred_y = int(np.argmax(proba))

    # Make sampling deterministic
    np.random.seed(SEED + int(idx))

    sv = explainer.shap_values(x, nsamples=NSAMPLES)
    # sv is a list: one array per class
    sv0 = np.array(sv[0], dtype=np.float32)
    sv1 = np.array(sv[1], dtype=np.float32)
    shap_class0.append(sv0)
    shap_class1.append(sv1)

    # Compact top-K by absolute value for JSON summary
    top = np.argsort(np.abs(sv1))[-K_TOP:][::-1]
    weights_top = [
        {"feature": feature_names[int(j)], "value": float(x[int(j)]), "shap": float(sv1[int(j)])}
        for j in top
    ]

    per_instance.append(
        {
            "test_index": int(idx),
            "true_y": true_y,
            "pred_y": pred_y,
            "proba": [float(p) for p in proba],
            "topK_class1": weights_top,
        }
    )

    # Waterfall plot for class 1
    try:
        exp_obj = shap.Explanation(
            values=sv1,
            base_values=float(explainer.expected_value[1]),
            data=x,
            feature_names=feature_names,
        )
        shap.plots.waterfall(exp_obj, max_display=K_TOP, show=False)
        plt.tight_layout()
        png_path = OUT_DIR / f"waterfall_test_{idx:03d}_class1.png"
        plt.savefig(png_path, dpi=150)
        plt.close()
    except Exception as e:
        per_instance[-1]["plot_error"] = str(e)

shap_class0 = np.stack(shap_class0, axis=0)
shap_class1 = np.stack(shap_class1, axis=0)

np.save(OUT_DIR / "shap_values_class0.npy", shap_class0)
np.save(OUT_DIR / "shap_values_class1.npy", shap_class1)

summary = {
    "seed": SEED,
    "nsamples": int(NSAMPLES),
    "k_top": int(K_TOP),
    "background_size": int(len(bg_idx)),
    "n_examples": int(len(test_indices)),
    "class_names": class_names,
    "results": per_instance,
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

(shap_class0.shape, shap_class1.shape)


((10, 2), (10, 2))